<a href="https://colab.research.google.com/github/BandanaSingha24/Integrated_Multiomic_Survival_Model/blob/main/01_Python_Foundation%20_for_Bioinformatics/new_processed_mutation_daa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

# Step 1: Load the raw mutation data
raw_mut_df = pd.read_csv('/content/data_mutations.txt', sep='\t', skiprows=1)

# Step 2: Filter out 'Silent' mutations to reduce noise
mut_cleaned = raw_mut_df[~raw_mut_df['Variant_Classification'].isin(['Silent'])]

# Step 3: Create a Binary Matrix using Patient IDs and Gene Symbols
# This aligns mutations to specific patients
final_matrix = pd.crosstab(mut_cleaned['Tumor_Sample_Barcode'], mut_cleaned['Hugo_Symbol'])

# Step 4: Ensure the matrix contains only 0 and 1
# 1 indicates presence of mutation, 0 indicates absence
final_binary_matrix = (final_matrix > 0).astype(int)

# Step 5: Check the shape to verify data integrity
print("Binary Matrix Shape:", final_binary_matrix.shape)
print(final_binary_matrix.head())

Binary Matrix Shape: (2358, 173)
Hugo_Symbol           ACVRL1  AFF2  AGMO  AGTR2  AHNAK  AHNAK2  AKAP9  AKT1  \
Tumor_Sample_Barcode                                                          
MB-0002                    0     0     0      0      0       0      0     0   
MB-0005                    0     0     0      0      0       0      0     0   
MB-0006                    0     0     0      0      0       0      0     0   
MB-0008                    0     0     0      0      0       0      0     0   
MB-0010                    0     0     0      0      0       0      0     0   

Hugo_Symbol           AKT2  ALK  ...  THADA  THSD7A  TP53  TTYH1  UBR5  USH2A  \
Tumor_Sample_Barcode             ...                                            
MB-0002                  0    0  ...      0       0     1      0     0      0   
MB-0005                  0    0  ...      0       0     0      0     0      0   
MB-0006                  0    0  ...      0       0     0      0     0      0   
MB-0008 

In [3]:
# step 2:mutation matrix cleanup

# 1. Reset index and rename to match clinical key exactly
mutation_df = final_binary_matrix.copy()
mutation_df.index.name = 'PATIENT_ID'
mutation_df = mutation_df.reset_index()

# 2. Clear out any residual axis name artifacts
mutation_df.columns.name = None

# 3. Drop genes with zero variance (never mutated in any patient)
# Select only gene columns (excluding PATIENT_ID)
gene_cols = [col for col in mutation_df.columns if col != 'PATIENT_ID']
zero_variance_genes = [col for col in gene_cols if mutation_df[col].var() == 0]

mutation_filtered = mutation_df.drop(columns=zero_variance_genes)
print(f"Dropped {len(zero_variance_genes)} genes with zero variance.")
print("Final mutation matrix shape:", mutation_filtered.shape)

Dropped 0 genes with zero variance.
Final mutation matrix shape: (2358, 174)


In [6]:
from google.colab import files

mutation_filtered.to_csv('md-1_processed_mutation_data.csv', index=False)

files.download('md-1_processed_mutation_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>